In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

import ast

In [0]:
dbutils.widgets.text('gold_params', '')

In [0]:
gold_params = dbutils.widgets.get('gold_params')

if not gold_params:
    raise Exception('Missing notebook parameters')

params = ast.literal_eval(gold_params)

table_name = params.get('table_name')
natural_key = params.get('natural_key')

table_name, natural_key

('categorias', ['natureza', 'categoriaPai', 'categoria'])

In [0]:
df_gold = (
    spark.read.table(f'mobills.silver.{table_name}')
    .select('id', *natural_key)
)

if natural_key:
    condicao_update = " OR ".join([f"NOT (src.{c} <=> tgt.{c})" for c in natural_key])
else:
    condicao_update = "1=0"

if not spark.catalog.tableExists(f'mobills.gold.{table_name}'):
    # só executa na primeira vez
    (
        df_gold
        .write
        .format('delta')
        .mode('append')
        .saveAsTable(f'mobills.gold.{table_name}')
    )
else:
    delta_gold = DeltaTable.forName(spark, f'mobills.gold.{table_name}')

    # merge into delta
    (
        delta_gold.alias('tgt')
        .merge(
            df_gold.alias('src'),
            'tgt.id = src.id'
        )
        .whenMatchedUpdateAll(condition=condicao_update)
        .whenNotMatchedInsertAll()
        .execute()
    )